# Extrapolate EN4 TF from regular grid to ISMIP
**NOTE:** This notebook will only need to be run once to produce the EN4 TF forcing on the ISMIP grid.

05 May 2026 | DF

### Setup

In [ ]:
## files
DirSave = f'./../gris-iceocean-outfiles'
en4_file = f'{DirSave}/tf-Hadley-1950_2020_regrid.nc'


### Imports

In [ ]:
# libraries
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import netCDF4 as nc
from netCDF4 import Dataset
import os
from pyproj import Transformer
from scipy.interpolate import interp1d
from datetime import datetime


In [ ]:
## load effective geometry (spatial look-up table)

# files
xy_eff_file = './XY_eff.nc'
z_eff_file = './z_eff.nc'

# effective geometry
# NB replace masked values with NaNs
X_eff = nc.Dataset(xy_eff_file).variables['X_eff'][:].filled(np.nan)
Y_eff = nc.Dataset(xy_eff_file).variables['Y_eff'][:].filled(np.nan)
z_eff = nc.Dataset(z_eff_file).variables['z_eff'][:].filled(np.nan)

# ismip coordinates at which effective geometry applies
x = nc.Dataset(xy_eff_file).variables['x'][:].filled(np.nan)
y = nc.Dataset(xy_eff_file).variables['y'][:].filled(np.nan)
X, Y = np.meshgrid(x, y)

# vertical grid of effective depths
z = np.flipud(np.unique(z_eff[z_eff<=0]))


In [ ]:
fig, ax= plt.subplots()
sc = ax.scatter(X_eff[::4, ::4], Y_eff[::4, ::4], c=z_eff[::4,::4])
plt.colorbar(sc, label='effective depth read in (m)')

Use xarray to load in a view of the dataset for future reference.

In [ ]:
## load and process data
ds_en4 = xr.open_dataset(en4_file, decode_times='timeDim')

# en4 model coordinates
lat_en4 = nc.Dataset(en4_file).variables['lat'][:].filled(np.nan)
lon_en4 = nc.Dataset(en4_file).variables['lon'][:].filled(np.nan)
z_en4 = (nc.Dataset(en4_file).variables['depth'][:].filled(np.nan)) ## QDM processed file should have depth in m
t_en4 = nc.Dataset(en4_file).variables['time'][:].filled(np.nan)

## Common grid has y, x in EPSG 3413 already
x_en4 = nc.Dataset(en4_file).variables['x'][:].filled(np.nan)
y_en4 = nc.Dataset(en4_file).variables['y'][:].filled(np.nan)

# ## load TF
## No masking, just set <0 to 0 and allow all other missing data -- should be missing where there is no ocean data in domain
#TF_en4_raw = nc.Dataset(en4_file).variables['TF'][:]
TF_en4_raw_ds = xr.open_dataset(en4_file)
TF_en4_raw_ds = TF_en4_raw_ds.transpose('depth','time','y','x')
TF_en4_raw = TF_en4_raw_ds['TF'].to_masked_array()
TF_en4_nonneg = np.ma.masked_less(TF_en4_raw, 0) ## filter out erroneous negative values

TF_en4 = TF_en4_raw
TF_en4[TF_en4<0] = 0

In [ ]:
## we can tell from the read-in above that TF_en4 from EN4 has dimensions (depth, time, y, x)
## make a filled version of the masked array for interpolation and shuffling
TF_en4_filled = np.ma.filled(TF_en4, fill_value=np.nan) ## filter out fill value
TF_nop_en4_orig_crop = TF_en4_filled.reshape(TF_en4_filled.shape[0], ## depth
                                          TF_en4_filled.shape[1], ## time?
                                          TF_en4_filled.shape[2]*TF_en4_filled.shape[3]) ## should be x*y
TF_nop_en4_orig_crop = np.ma.masked_greater(TF_nop_en4_orig_crop, 1.1e10)

xm, ym = np.meshgrid(x_en4, y_en4)
x_mod = xm.flatten()
y_mod = ym.flatten()

# get max depth at which each en4 point has a thermal forcing value
depth_en4 = np.full(x_mod.shape, np.nan)
for i in range(len(x_mod)):
    tf = TF_nop_en4_orig_crop[:,0,i] ## careful with order of axes here
    if np.all(np.isnan(tf)):
        continue
    elif np.all(~np.isnan(tf)):
        depth_en4[i] = z[-1]
    else:
        first_nan = np.argmax(np.isnan(tf))
        depth_en4[i] = z[first_nan-1]
        

In [ ]:
## plots to check things make sense

fig, ax = plt.subplots()
# ax.contourf(lon_en4, lat_en4, TF_en4[0,0, :,:]) ## for raw
# ax.contourf(lon_en4, lat_en4, TF_en4[0,:,:,0]) ## for QDM-processed
ax.contourf(lon_en4, lat_en4, TF_en4[0,0,:,:]) ## for mean-corrected
plt.show()

# plt.scatter(xm, ym, c=TF_nop_en4[0,0,:])
# plt.colorbar(label='thermal forcing at first time and depth slice (°C)')
# plt.axis('equal')
# plt.show()

plt.scatter(x_mod, y_mod, c=depth_en4)
plt.colorbar(label='max depth of non-NaN data in EN4 (m)')
plt.axis('equal')
plt.show()


In [ ]:
## match ismip points to en4 points

# linear versions of effective position arrays
X_eff_flat = X_eff.flatten()
Y_eff_flat = Y_eff.flatten()

# initialise arrays that will store the linear index of the required en4 point
x_ind = np.full_like(X_eff_flat, np.nan, dtype=float)
z_ind = np.full_like(X_eff_flat, np.nan, dtype=float)

# loop over effective depth grid
for k, z_val in enumerate(z):
    
    # get all ismip points with this effective depth
    i_inds = np.where(z_eff.flatten() == z_val)[0]

    # save the effective depth index
    z_ind[i_inds] = k

    # get all en4 points that have data at this depth
    c_inds = np.where(depth_en4.flatten() <= z_val)[0]

    # loop over ismip points with this effective depth and find closest en4 point to effective position
    for i in i_inds:
        dsq = (x_mod[c_inds] - X_eff_flat[i])**2 + (y_mod[c_inds] - Y_eff_flat[i])**2
        id_min = np.argmin(dsq)
        x_ind[i] = c_inds[id_min]
    

In [ ]:
## now we fill the ismip TF using the matched en4 positions and depths

# initialise array
TF_nop = np.full((len(t_en4), len(x_ind)), np.nan)

# fill array knowing that the ismip TF(t,i) equals the en4 TF(z_ind(i),t,x_ind(i))
for i in range(len(x_ind)):
    if not np.isnan(x_ind[i]):
        ## depth first
        TF_nop[:,i] = TF_nop_en4_orig_crop[int(z_ind[i]), :, int(x_ind[i])]

# reshape array for consistency with ismip coordinates
TF = TF_nop.reshape(len(t_en4), X.shape[0], X.shape[1])


In [ ]:
# # check if any TF are less than 0 and fix
# TF = np.full_like(TF_nop, np.nan, dtype=float) ## no pressure correction
inds = np.where(TF < 0)
num_corrected = len(inds[0])
if num_corrected > 0:
    TF[inds] = 0
    print(f"Warning: {num_corrected} pixels corrected for TF<0")

# all the unconnected below sea level points must be assigned TF=0 -  these points have z_eff=NaN
TF[:, np.isnan(z_eff)] = 0

In [ ]:
## plot an example of the result

plt.pcolormesh(X, Y, TF[0, :, :])
cbar = plt.colorbar()
cbar.set_label('extrapolated thermal forcing at first time slice (°C)')
plt.axis('equal')
plt.show()

## Write out
Use xarray to write nice metadata and keep date format intact.  These cells prep the data to write, then there are two separate options to write a single or batched file.  The choice of `output_option` is set in the header of this notebook.

In [ ]:
TF_out_ds = xr.Dataset(
    data_vars = dict(TF=(['time', 'y', 'x'], TF)), 
    coords = dict(
        time = ds_en4.time, ## cheat a bit and take the datetime64 index from ds_en4, read in above
        y = y,
        x = x)
)

TF_out_ds

In [ ]:
TF_out_ds = TF_out_ds.convert_calendar('noleap')
TF_out_ds

Explicitly set some metadata, and specify that TF should be written as float32 (like Donald did in his approach).

In [ ]:
TF_out_ds.coords['y'].attrs['units'] = 'meters'
TF_out_ds.coords['y'].attrs['projection'] = 'EPSG:3413'
TF_out_ds.coords['x'].attrs['units'] = 'meters'
TF_out_ds.coords['x'].attrs['projection'] = 'EPSG:3413'

TF_out_ds.TF.attrs['long_name'] = 'Ocean thermal forcing'
TF_out_ds.TF.attrs['fill_value'] = 1.1e20 ## list fillvalue even though we have not done explicit filling in this step
TF_out_ds

In [ ]:
TF_reduced = TF_out_ds.astype('float32')
TF_reduced

### Write the output file

In [ ]:
## SINGLE OUTPUT FILE
## write to output file
from datetime import datetime

now = datetime.now()

## Make filename tags showing time for the output 
# ## this is very janky, but datetime64 objects are stubborn
# FirstYear = TF_reduced.time[0].values.astype(str).split('-')[0] 
# LastYear = TF_reduced.time[-1].values.astype(str).split('-')[0]
## with cftime axes:
FirstYear = TF_reduced.time[0].values.item().year
LastYear = TF_reduced.time[-1].values.item().year

p_tag = 'PFromStep1'

SelModel='EN4'

## file name
out_fn = DirSave + '/TF_MeanCorrected-ISMIP_Grid-{}-{}_{}-{}-{}.nc'.format(SelModel, 
                                                    FirstYear, LastYear, 
                                                    p_tag, 
                                                    now.strftime('%Y%m%d'))

# ds_temp = tf_out.to_dataset(name='TF')
ds_out = TF_reduced.assign_attrs(title='Ocean thermal forcing for {}'.format(SelModel),
                             summary='TF computed following Verjans bias correction and Slater inland mapping,' + 
                             'in a bounding box around Greenland, for ISMIP7 Greenland forcing.' +
                                ' Process code: github.com/ehultee/gris-iceocean-process',
                             institution='NASA Goddard Space Flight Center',
                             creation_date=now.strftime('%Y-%m-%d %H:%M:%S'))

## write it!
ds_out.to_netcdf(path=out_fn,
                encoding={'TF': {'zlib': True, 'complevel':9}}) ## set compression level

In [ ]:
ds_out.TF.time[5]

In [ ]:
(ds_out.TF[-7]-ds_out.TF[5]).plot()

In [ ]:
if output_option == 'batch':
    ## MULTI OUTPUT FILE
    # from dask.diagnostics import ProgressBar

    TF_reduced_hist = TF_reduced.sel(time=time_slice_hist)
    TF_reduced_proj = TF_reduced.sel(time=time_slice_proj)

    now = datetime.now()
    
    to_write = [TF_reduced_hist,TF_reduced_proj]
    
    for d in to_write:
        print('Processing...')
        ## Make filename tags showing time for the output 
        # ## this is very janky, but datetime64 objects are stubborn
        # FirstYear = d.time[0].values.astype(str).split('-')[0] 
        # LastYear = d.time[-1].values.astype(str).split('-')[0]
        ## with cftime axes:
        FirstYear = d.time[0].values.item().year
        LastYear = d.time[-1].values.item().year
        
        # FirstYear=2015
        # LastYear=2100
        # ##REMEMBER TO CHANGE BACK!  This one only for CESM raw
        
        if PressureAlreadyIncluded:
            p_tag = 'PFromStep1'
        else:
            p_tag = 'PCorrected_Step3'
        
        SelModel='CESM2-WACCM'
        
        ## file name
        out_fn = DirSave + '/' + 'TF_MeanCorrected-ISMIP_Grid-{}-{}_{}-{}-{}.nc'.format(SelModel, 
                                                            FirstYear, LastYear, 
                                                            p_tag, 
                                                            now.strftime('%Y%m%d'))
        print('Assigning attrs')
        # ds_temp = tf_out.to_dataset(name='TF')
        ds_out = d.assign_attrs(title='Ocean thermal forcing for {}'.format(SelModel),
                                     summary='TF computed following Verjans bias correction and Slater inland mapping,' + 
                                     'in a bounding box around Greenland, for ISMIP7 Greenland forcing.' +
                                        ' Mean correction based on EN4, 1985-2014.' +
                                        ' Process code: github.com/ehultee/gris-iceocean-process',
                                     institution='NASA Goddard Space Flight Center',
                                     creation_date=now.strftime('%Y-%m-%d %H:%M:%S'))
    
        print('Writing: ' + out_fn)
        ## write it!
        # with ProgressBar():
        ds_out.to_netcdf(path=out_fn,
                        encoding={'TF': {'zlib': True, 'complevel':9}}) ## set compression level
    
    print('Done!')